# 02 - Source Loading and Metadata Extraction

**ChatGPT Track**  
**allen-lab-report-tool**

This notebook creates a source metadata artifact and connects it to the Allen Lab context profile from Notebook 01.

Notebook 02 keeps the pipeline simple:

```text
source record
→ metadata validation
→ Allen context matching
→ JSON + Markdown export
```

Expected repo layout:

```text
allen-lab-report-tool/
├── src/
│   └── chatgpt/
│       └── lab_context.py
├── results/
│   └── chatgpt/
│       └── allen_lab_context.json
└── notebooks/
    └── chatgpt/
        └── 02_source_loading_and_metadata.ipynb
```


In [ ]:
# ================================================
# SETUP: Colab + local
# ================================================
from pathlib import Path
import json
import sys
import subprocess
import importlib.util
from datetime import datetime, timezone

REPO_NAME = "allen-lab-report-tool"
REPO_URL = "https://github.com/thinkthoughts/allen-lab-report-tool.git"

cwd = Path.cwd()

# Case 1: running from repo root
if (cwd / "src" / "chatgpt" / "lab_context.py").exists():
    repo_root = cwd

# Case 2: running from notebooks/chatgpt inside repo
elif cwd.name == "chatgpt" and cwd.parent.name == "notebooks":
    repo_root = cwd.parents[1]

# Case 3: Colab default /content with cloned repo
elif (cwd / REPO_NAME / "src" / "chatgpt" / "lab_context.py").exists():
    repo_root = cwd / REPO_NAME

# Case 4: Colab default /content with no repo cloned yet
else:
    print("Repo not found in current runtime. Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo_root = cwd / REPO_NAME

src_path = repo_root / "src"
chatgpt_path = src_path / "chatgpt"
lab_context_path = chatgpt_path / "lab_context.py"

print("cwd:", cwd)
print("repo_root:", repo_root)
print("src_path:", src_path)
print("chatgpt_path:", chatgpt_path)
print("lab_context exists:", lab_context_path.exists())

if not lab_context_path.exists():
    raise FileNotFoundError(f"Missing expected file: {lab_context_path}")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chatgpt.lab_context import ALLEN_LAB_CONTEXT

print("ChatGPT lab context loaded.")
print("Institution:", ALLEN_LAB_CONTEXT["institution"])


## 1. Load Notebook 01 Context Artifact

Notebook 02 first tries to load:

```text
results/chatgpt/allen_lab_context.json
```

If that file does not exist yet, it falls back to `src/chatgpt/lab_context.py`.


In [ ]:
results_dir = repo_root / "results" / "chatgpt"
reports_dir = repo_root / "reports" / "chatgpt"

results_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

context_json_path = results_dir / "allen_lab_context.json"

if context_json_path.exists():
    allen_context = json.loads(context_json_path.read_text(encoding="utf-8"))
    print("Loaded context artifact:", context_json_path)
else:
    allen_context = {
        **ALLEN_LAB_CONTEXT,
        "generator_track": "chatgpt",
        "source_file": "src/chatgpt/lab_context.py",
    }
    print("Notebook 01 context artifact not found; using ALLEN_LAB_CONTEXT from src.")

allen_context


## 2. Define Example Source Record

This placeholder can be replaced later with a specific Allen Lab paper, dataset page, method note, or seminar source.

Notebook 02 tests the metadata path before adding full PDF parsing or web extraction.


In [ ]:
SOURCE_RECORD = {
    "title": "Example Allen Lab Source for Report-Tool Development",
    "authors": ["Allen Lab / Allen Institute source placeholder"],
    "source_type": "paper_or_dataset_note",
    "institution": "Allen Institute",
    "abstract_or_summary": (
        "This placeholder source represents an Allen Lab research output "
        "used to test context-aware metadata extraction for lab report generation. "
        "It includes terms such as cell types, brain atlas, single-cell sequencing, "
        "open science, dataset provenance, methods traceability, and visualization-ready summaries."
    ),
    "keywords": [
        "cell types",
        "brain atlas",
        "single-cell sequencing",
        "open science",
        "dataset provenance",
        "methods traceability",
        "visualization-ready summaries",
    ],
    "source_url": "",
    "notes": [
        "Replace this placeholder with a specific paper, dataset page, or methods source.",
        "Notebook 02 tests metadata structure before full document parsing.",
    ],
}

SOURCE_RECORD


## 3. Validate Source Metadata

In [ ]:
required_source_keys = [
    "title",
    "authors",
    "source_type",
    "institution",
    "abstract_or_summary",
    "keywords",
]

missing_source_keys = [key for key in required_source_keys if key not in SOURCE_RECORD]

if missing_source_keys:
    raise ValueError(f"Missing required source keys: {missing_source_keys}")

if not isinstance(SOURCE_RECORD["authors"], list):
    raise TypeError("SOURCE_RECORD['authors'] must be a list.")

if not isinstance(SOURCE_RECORD["keywords"], list):
    raise TypeError("SOURCE_RECORD['keywords'] must be a list.")

for key in required_source_keys:
    print(f"✓ {key}: {type(SOURCE_RECORD[key]).__name__}")


## 4. Display Source Metadata

In [ ]:
import pandas as pd

source_overview_df = pd.DataFrame([
    {"field": "title", "value": SOURCE_RECORD["title"]},
    {"field": "source_type", "value": SOURCE_RECORD["source_type"]},
    {"field": "institution", "value": SOURCE_RECORD["institution"]},
    {"field": "source_url", "value": SOURCE_RECORD["source_url"] or "(not set yet)"},
])

authors_df = pd.DataFrame({"author": SOURCE_RECORD["authors"]})
keywords_df = pd.DataFrame({"keyword": SOURCE_RECORD["keywords"]})
notes_df = pd.DataFrame({"note": SOURCE_RECORD["notes"]})

display(source_overview_df)
display(authors_df)
display(keywords_df)
display(notes_df)


## 5. Match Source Terms to Allen Context

This simple matcher checks overlap between source text/keywords and the Allen context profile.

Later notebooks can replace this with richer parsing, embeddings, or manual review.


In [ ]:
def normalize_text(text):
    return str(text).lower().replace("-", " ")

def contains_term(haystack, term):
    return normalize_text(term) in normalize_text(haystack)

source_text = " ".join([
    SOURCE_RECORD["title"],
    SOURCE_RECORD["abstract_or_summary"],
    " ".join(SOURCE_RECORD["keywords"]),
    " ".join(SOURCE_RECORD.get("notes", [])),
])

matched_focus_areas = [
    term for term in allen_context.get("likely_focus_areas", [])
    if contains_term(source_text, term)
]

matched_equipment_or_platforms = [
    term for term in allen_context.get("likely_equipment_or_platforms", [])
    if contains_term(source_text, term)
]

matched_report_priorities = [
    term for term in allen_context.get("report_priorities", [])
    if contains_term(source_text, term)
]

matches = {
    "matched_focus_areas": matched_focus_areas,
    "matched_equipment_or_platforms": matched_equipment_or_platforms,
    "matched_report_priorities": matched_report_priorities,
}

matches


## 6. Display Context Matches

In [ ]:
matches_df = pd.concat(
    [
        pd.DataFrame({"match_type": "focus_area", "term": matched_focus_areas}),
        pd.DataFrame({"match_type": "equipment_or_platform", "term": matched_equipment_or_platforms}),
        pd.DataFrame({"match_type": "report_priority", "term": matched_report_priorities}),
    ],
    ignore_index=True,
)

if matches_df.empty:
    print("No direct context matches found.")
else:
    display(matches_df)


## 7. Build Metadata Artifact

In [ ]:
metadata_artifact = {
    "generator_track": "chatgpt",
    "source_file": "notebooks/chatgpt/02_source_loading_and_metadata.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_record": SOURCE_RECORD,
    "allen_context_reference": {
        "institution": allen_context.get("institution", "Allen Institute"),
        "context_source": str(context_json_path.relative_to(repo_root)) if context_json_path.exists() else "src/chatgpt/lab_context.py",
    },
    "context_matches": matches,
}

metadata_artifact


## 8. Export JSON

In [ ]:
metadata_json_path = results_dir / "source_metadata.json"

with metadata_json_path.open("w", encoding="utf-8") as f:
    json.dump(metadata_artifact, f, indent=2, ensure_ascii=False)

print("Wrote:", metadata_json_path)


## 9. Export Markdown

In [ ]:
def bullets(items):
    if not items:
        return "- (none)"
    return "\n".join(f"- {item}" for item in items)

source = metadata_artifact["source_record"]
match = metadata_artifact["context_matches"]

metadata_md = f"""# Source Metadata

**Generator track:** ChatGPT  
**Notebook:** `notebooks/chatgpt/02_source_loading_and_metadata.ipynb`  
**Context:** {metadata_artifact['allen_context_reference']['institution']}

## Source

**Title:** {source['title']}  
**Source type:** {source['source_type']}  
**Institution:** {source['institution']}  
**Source URL:** {source['source_url'] or '(not set yet)'}

## Authors / Source Attribution

{bullets(source['authors'])}

## Abstract or Summary

{source['abstract_or_summary']}

## Keywords

{bullets(source['keywords'])}

## Notes

{bullets(source.get('notes', []))}

## Matched Allen Context Terms

### Focus areas

{bullets(match['matched_focus_areas'])}

### Equipment or platforms

{bullets(match['matched_equipment_or_platforms'])}

### Report priorities

{bullets(match['matched_report_priorities'])}

## Next Step

Notebook 03 can convert this metadata artifact into context-aware report sections.
"""

metadata_md_path = reports_dir / "source_metadata.md"
metadata_md_path.write_text(metadata_md, encoding="utf-8")

print("Wrote:", metadata_md_path)


## 10. Confirm Exports

In [ ]:
print(metadata_json_path.read_text(encoding="utf-8"))


In [ ]:
print(metadata_md_path.read_text(encoding="utf-8"))


## 11. Summary

Notebook 02 creates the first source metadata artifact for the ChatGPT track.

Current outputs:

```text
results/chatgpt/source_metadata.json
reports/chatgpt/source_metadata.md
```

**Next:** Notebook 03 — Context-Aware Report Sections.
